In [1]:
# Imports
from poolparty.counter import (
    Counter, CounterManager,
    multiply_counters, sum_counters, synchronize_counters, slice_counter, repeat_counter, shuffle_counter, interleave_counters
)


In [2]:
with CounterManager() as mgr:
    A = Counter(num_states=3, name='A')  
    B = Counter(num_states=3, name='B')  
    C = Counter(num_states=3, name='C') 
    X = interleave_counters(A,B,C, name='X')
    Y = interleave_counters(C,B,A, name='Y')
    mgr.test_iteration(X)
    mgr.test_iteration(Y)
    
    print('X tree:')
    X.print_tree()
    print('Y tree:')
    Y.print_tree()
    print('mgr graph:')
    mgr.print_graph()

,A.state,B.state,C.state,X.state,Y.state
X.state,,,,,
0,0,-1,-1,0,-1
1,-1,0,-1,1,-1
2,-1,-1,0,2,-1
3,1,-1,-1,3,-1
4,-1,1,-1,4,-1
5,-1,-1,1,5,-1
6,2,-1,-1,6,-1
7,-1,2,-1,7,-1
8,-1,-1,2,8,-1


,A.state,B.state,C.state,X.state,Y.state
Y.state,,,,,
0,-1,-1,0,-1,0
1,-1,0,-1,-1,1
2,0,-1,-1,-1,2
3,-1,-1,1,-1,3
4,-1,1,-1,-1,4
5,1,-1,-1,-1,5
6,-1,-1,2,-1,6
7,-1,2,-1,-1,7
8,2,-1,-1,-1,8


X tree:
X [Interleave, n=9]
├── A [Leaf, n=3]
├── B [Leaf, n=3]
└── C [Leaf, n=3]
Y tree:
Y [Interleave, n=9]
├── C [Leaf, n=3]
├── B [Leaf, n=3]
└── A [Leaf, n=3]
mgr graph:
X [Interleave, n=9]
├── A [Leaf, n=3]
├── B [Leaf, n=3]
└── C [Leaf, n=3]

Y [Interleave, n=9]
├── C [Leaf, n=3]
├── B [Leaf, n=3]
└── A [Leaf, n=3]


In [3]:
# Complicated Counter tree: ((A * B) + C) * D with slices and repeats

with CounterManager() as mgr:
    # Basic counters
    A = Counter(2).named('A')
    B = Counter(4).named('B')
    C = (A*B).named('C')
    D = Counter(3).named('D')
    E = Counter(5).named('E')
    F = (D+E).named('F')
    G = synchronize_counters(C, F).named('G')
    H = G[::-2].named('H')
    mgr.test_iteration(G)
    mgr.print_graph()

mgr.get_by_name('G')

,A.state,B.state,C.state,D.state,E.state,F.state,G.state,H.state
G.state,,,,,,,,
0,0,0,0,0,-1,0,0,-1
1,1,0,1,1,-1,1,1,-1
2,0,1,2,2,-1,2,2,-1
3,1,1,3,-1,0,3,3,-1
4,0,2,4,-1,1,4,4,-1
5,1,2,5,-1,2,5,5,-1
6,0,3,6,-1,3,6,6,-1
7,1,3,7,-1,4,7,7,-1


H [Slice, n=4]
└── G [Synchronize, n=8]
    ├── C [Multiply, n=8]
    │   ├── A [Leaf, n=2]
    │   └── B [Leaf, n=4]
    └── F [Sum, n=8]
        ├── D [Leaf, n=3]
        └── E [Leaf, n=5]


Counter(name='G', id=6, op=SynchronizeCoOp, num_states=8, state=-1)

In [4]:
# Basic Counter - iterate through states
with CounterManager() as mgr:
    A = Counter(4, name='A')
    mgr.test_iteration(A)


,A.state
A.state,
0,0
1,1
2,2
3,3


In [5]:
# MultiplyCoOp - Cartesian product (A * B)
with CounterManager() as mgr:
    A = Counter(2, name='A')
    B = Counter(3, name='B')
    C = A * B
    C.name = 'C'
    mgr.test_iteration(C)


,A.state,B.state,C.state
C.state,,,
0,0,0,0
1,1,0,1
2,0,1,2
3,1,1,3
4,0,2,4
5,1,2,5


In [6]:
# SumCoOp - Disjoint union (A + B), inactive branches show -1
with CounterManager() as mgr:
    A = Counter(2, name='A')
    B = Counter(3, name='B')
    C = A + B
    C.name = 'C'
    mgr.test_iteration(C)


,A.state,B.state,C.state
C.state,,,
0,0,-1,0
1,1,-1,1
2,-1,0,2
3,-1,1,3
4,-1,2,4


In [7]:
# SynchronizeCoOp - Keep counters in lockstep
with CounterManager() as mgr:
    A = Counter(4, name='A')
    B = Counter(4, name='B')
    S = synchronize_counters(A, B, name='S')
    mgr.test_iteration(S)


,A.state,B.state,S.state
S.state,,,
0,0,0,0
1,1,1,1
2,2,2,2
3,3,3,3


In [8]:
# SliceCoOp - Select subset of states (A[::2] even, A[::-1] reversed)
with CounterManager() as mgr:
    A = Counter(10, name='A')
    B = A[3:9:2]
    B.name = 'B'
    mgr.test_iteration(B)


,A.state,B.state
B.state,,
0,3,0
1,5,1
2,7,2


In [9]:
# SliceCoOp - Reversed iteration (A[::-1])
with CounterManager() as mgr:
    A = Counter(10, name='A')
    R = A[-3:2:-1]
    R.name = 'R'
    mgr.test_iteration(R)


,A.state,R.state
R.state,,
0,7,0
1,6,1
2,5,2
3,4,3
4,3,4


In [10]:
# RepeatCoOp - Repeat counter N times (A * 3)
with CounterManager() as mgr:
    A = Counter(2, name='A')
    R = A * 3
    R.name = 'R'
    mgr.test_iteration(R)


,A.state,R.state
R.state,,
0,0,0
1,1,1
2,0,2
3,1,3
4,0,4
5,1,5


In [11]:
# ShuffleCoOp - Randomly permute states (reproducible with seed)
with CounterManager() as mgr:
    A = Counter(6, name='A')
    S = shuffle_counter(A, seed=42, name='S')
    mgr.test_iteration(S)


,A.state,S.state
S.state,,
0,3,0
1,1,1
2,2,2
3,4,3
4,0,4
5,5,5


In [12]:
# split_counter - Split one counter into multiple counters
from poolparty.counter import split_counter

with CounterManager() as mgr:
    A = Counter(10, name='A')
    left, mid, right = split_counter(A, 3, names=['left', 'mid', 'right'])
    mgr.test_iteration(left)  # mid covers A states 4-6
    mgr.test_iteration(mid)
    mgr.test_iteration(right)


,A.state,left.state,mid.state,right.state
left.state,,,,
0,0,0,-1,-1
1,1,1,-1,-1
2,2,2,-1,-1
3,3,3,-1,-1


,A.state,left.state,mid.state,right.state
mid.state,,,,
0,4,-1,0,-1
1,5,-1,1,-1
2,6,-1,2,-1


,A.state,left.state,mid.state,right.state
right.state,,,,
0,7,-1,-1,0
1,8,-1,-1,1
2,9,-1,-1,2


In [13]:
# InterleaveCoOp - Alternate states across counters (requires same num_states)
with CounterManager() as mgr:
    A = Counter(3, name='A')
    B = Counter(3, name='B')
    I = interleave_counters(A, B, name='I')
    mgr.test_iteration(I)

,A.state,B.state,I.state
I.state,,,
0,0,-1,0
1,-1,0,1
2,1,-1,2
3,-1,1,3
4,2,-1,4
5,-1,2,5
